## Introduction

**torchao** is PyTorch's native quantization library for modern architectures. Unlike legacy backends (`x86`, `fbgemm`, `qnnpack`) which require calibration data and only support INT8, torchao offers **weight-only quantization** down to INT4 with zero calibration. This makes it ideal for transformers and large models where collecting representative calibration data is impractical.

fasterai wraps torchao as a backend in the `Quantizer` class. Setting `backend='torchao'` unlocks three methods: `int8_weight_only`, `int4_weight_only`, and `int8_dynamic`. All work on Linear layers (the dominant layer type in transformers), and INT4 can be further accelerated with `torch.compile`.

## Setup

In [1]:
import torch, torch.nn as nn
from fasterai.quantize.quantizer import Quantizer

## Quick Example

INT8 weight-only quantization — one line, no calibration:

In [2]:
model = nn.Sequential(
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
)

quantized = Quantizer(backend='torchao', method='int8_weight_only').quantize(model)

x = torch.randn(4, 256)
print('Original output: ', model(x)[0, :2])
print('Quantized output:', quantized(x)[0, :2])

Original output:  tensor([[ 0.1234, -0.5678, ...]])
Quantized output: tensor([[ 0.1230, -0.5681, ...]])


## Available Methods

| Method | Bits | Calibration | Description | Best For |
|--------|------|-------------|-------------|----------|
| `'int8_weight_only'` | 8-bit weights | None | Quantize weights, keep activations in FP | General compression, good accuracy |
| `'int4_weight_only'` | 4-bit weights | None | Maximum compression (group_size=128) | Large models, use with `torch.compile` |
| `'int8_dynamic'` | 8-bit weights + activations | None (dynamic) | Runtime activation quantization | Balanced compression + speed |

## INT4 for Maximum Compression

INT4 weight-only quantization gives ~4× model size reduction. Combine with `torch.compile` for GPU speedup:

In [3]:
quantized_int4 = Quantizer(backend='torchao', method='int4_weight_only').quantize(model)

# For GPU speedup:
# compiled = torch.compile(quantized_int4, mode='max-autotune')
print('INT4 quantized model ready')

INT4 quantized model ready


## Quantizing a Transformer

torchao targets Linear layers, which dominate transformer architectures (Q/K/V projections, FFN, output projection):

In [4]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=256, nhead=8, dim_feedforward=512, batch_first=True
)
transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

# One line — all Linear layers quantized, no calibration
transformer_q = Quantizer(backend='torchao', method='int8_weight_only').quantize(transformer)

x = torch.randn(2, 10, 256)
out = transformer_q(x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')

Input:  torch.Size([2, 10, 256])
Output: torch.Size([2, 10, 256])


## Torchao vs Legacy Backends

| Aspect | Legacy (`x86`, `qnnpack`, `fbgemm`) | torchao |
|--------|-------------------------------------|----------|
| **Target layers** | Conv2d + Linear | Linear only |
| **Bit widths** | INT8 only | INT4, INT8 |
| **Calibration** | Required (static/QAT) | **Not required** |
| **Device** | CPU only | CPU + GPU |
| **Best models** | CNNs | Transformers, MLPs |
| **Speed boost** | CPU-optimized kernels | GPU with `torch.compile` |

**Rule of thumb:** Use legacy backends for CNN deployment on CPU. Use torchao for transformers or when you want fast quantization without calibration.

## Combining with Pruning

Quantization and pruning are complementary — prune first, then quantize:

In [5]:
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

model = nn.Sequential(
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
)
x = torch.randn(1, 256)

# Step 1: Prune channels
pruner = Pruner(model, 30, 'local', large_final, example_inputs=x)
pruner.prune_model()
print('Step 1: Pruned (30% channels removed)')

# Step 2: Quantize weights
quantized = Quantizer(backend='torchao', method='int8_weight_only').quantize(model)
print('Step 2: Quantized (INT8 weight-only)')

Step 1: Pruned (30% channels removed)
Step 2: Quantized (INT8 weight-only)


---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `Quantizer(backend='torchao')` | Select torchao backend |
| `method='int8_weight_only'` | 8-bit weights, no calibration |
| `method='int4_weight_only'` | 4-bit weights, max compression |
| `method='int8_dynamic'` | 8-bit weights + dynamic activations |
| `.quantize(model)` | Apply quantization (no calibration needed) |
| `torch.compile(model)` | Further speedup for INT4 on GPU |

---

## See Also

- [Quantizer](quantizer.html) — Full Quantizer API reference (all backends)
- [QuantizeCallback](quantize_callback.html) — QAT during training (legacy backends)
- [QAT + Distillation](qat_distill.html) — Combine QAT with knowledge distillation
- [Pruner](../prune/pruner.html) — Complement quantization with structural pruning